# NSEMI Capstone — Script 2: Merge Datasets (1 per RQ)

**National Semiconductor Ecosystem Maturity Index (NSEMI)**  
Avinash Kashi Venugopal | Walsh College | QM640 | April 2026

This notebook merges the individual CSV files extracted in Script 1 into **one analysis-ready dataset per Research Question.**

| RQ | Merge Key | Technique | Output |
|----|-----------|-----------|--------|
| RQ1 | year_month | Temporal join (aggregate DGFT → monthly, merge with IIP) | `rq1_panel.csv` |
| RQ2 | state_name | Spatial join (CEA + SERC + weather + ISM status) | `rq2_panel.csv` |
| RQ3 | country + year | Entity-year join (cross-country) + state join (India) | `rq3_crosscountry.csv` + `rq3_india.csv` |
| RQ4 | country + year | Country-year panel join (WB + tax + electricity + subsidy) | `rq4_panel.csv` |

> Run this AFTER Script 1 has completed and CSVs are in `nsemi_data/raw/` and `nsemi_data/processed/`.


## 0. Setup


In [ ]:
# ─── Mount Google Drive ───
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path("/content/drive/MyDrive/NSEMI_Capstone")
print(f"✓ Drive mounted: {DRIVE_BASE}")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil, warnings
warnings.filterwarnings('ignore')

# ─── Use Google Drive paths (persistent) ───
DRIVE_BASE = Path("/content/drive/MyDrive/NSEMI_Capstone")
RAW = DRIVE_BASE / "raw"
PROC = DRIVE_BASE / "processed"
OUT = DRIVE_BASE / "analysis_ready"
OUT.mkdir(parents=True, exist_ok=True)

# Also create local workspace for fast I/O
LOCAL_OUT = Path("nsemi_data/analysis_ready")
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

# Verify data exists on Drive
raw_files = sorted(RAW.glob("*.csv")) if RAW.exists() else []
proc_files = sorted(PROC.glob("*.csv")) if PROC.exists() else []

if not raw_files and not proc_files:
    print("✗ No data found on Drive. Run Script 1 first!")
    print(f"  Expected at: {RAW}")
else:
    print(f"✓ Found {len(raw_files)} raw + {len(proc_files)} processed CSVs on Drive")
    print(f"\nAvailable raw CSVs:")
    for f in raw_files:
        print(f"  {f.name:55s} {len(pd.read_csv(f, low_memory=False)):>6} rows")
    if proc_files:
        print(f"\nAvailable processed CSVs:")
        for f in proc_files:
            print(f"  {f.name:55s} {len(pd.read_csv(f, low_memory=False)):>6} rows")


---
## RQ1: Ecosystem Depth — Temporal Join

**Goal:** 1 row per month (Apr 2017 – Mar 2026 = ~96 months)  
**Columns:** year_month, financial_year, import_usd_mn per HS segment, HHI per segment, IIP_div26, meity_production

**Merge strategy:**
1. DGFT (20,457 rows): aggregate country-level → monthly totals per HS code, then pivot to wide format
2. MOSPI IIP (481 rows): filter to Division 26 monthly index, extract year_month key  
3. MeitY (9 rows): annual → forward-fill to monthly
4. Comtrade (2,818 rows): kept as separate cross-country benchmarking panel


In [ ]:
# ─── RQ1: Merge into monthly time-series panel ───

# 1. DGFT: aggregate to monthly totals per HS code
dgft = pd.read_csv(RAW / "rq1_dgft_tradestat.csv")
print(f"DGFT raw: {dgft.shape}")

# Create year_month key
dgft["year_month"] = dgft["year"].astype(str) + "-" + dgft["month"].astype(str).str.zfill(2)

# Aggregate: sum import values across countries, per HS code per month
dgft_monthly = dgft.groupby(["year_month", "fiscal_year", "hs_code", "segment"]).agg(
    import_usd_mn=("import_value_usd_million", "sum"),
    n_countries=("country", "nunique"),
).reset_index()

# Pivot to wide: one column per HS segment
dgft_wide = dgft_monthly.pivot_table(
    index=["year_month", "fiscal_year"],
    columns="segment",
    values="import_usd_mn",
    aggfunc="sum"
).reset_index()
dgft_wide.columns.name = None

# Rename columns to import_usd_mn_* format
rename_map = {}
for col in dgft_wide.columns:
    if col not in ["year_month", "fiscal_year"]:
        rename_map[col] = f"import_usd_mn_{col.replace(' ', '_')}"
dgft_wide = dgft_wide.rename(columns=rename_map)

# Calculate HHI per segment per month (country concentration)
hhi_list = []
for (ym, fy, hs, seg), grp in dgft.groupby(["year_month", "fiscal_year", "hs_code", "segment"]):
    total = grp["import_value_usd_million"].sum()
    if total > 0:
        shares = grp.groupby("country")["import_value_usd_million"].sum() / total
        hhi = (shares ** 2).sum() * 10000
    else:
        hhi = 0
    hhi_list.append({"year_month": ym, f"hhi_{seg.replace(' ', '_')}": round(hhi, 1)})

hhi_df = pd.DataFrame(hhi_list)
# Pivot HHI
hhi_wide = hhi_df.groupby("year_month").first().reset_index()

print(f"DGFT monthly wide: {dgft_wide.shape}")
print(f"HHI wide: {hhi_wide.shape}")

# 2. MOSPI IIP: extract Division 26 monthly index
mospi = pd.read_csv(RAW / "rq1_mospi_iip_div26.csv", low_memory=False)
# Filter to Division 26 rows (NIC 2008 code 26)
iip_26 = mospi[mospi["_nic_2008"].astype(str).str.strip() == "26"].copy()
if len(iip_26) == 0:
    # Try alternate: look for "Computer, electronic" in description
    iip_26 = mospi[mospi["description"].str.contains("Computer|Electronic|Optical", case=False, na=False)].copy()

# If monthly data available
if "month" in mospi.columns and "_year" in mospi.columns:
    mospi_monthly = mospi[mospi["month"].notna()].copy()
    mospi_monthly["year_month"] = mospi_monthly["_year"].astype(str).str[:4] + "-" + mospi_monthly["month"].astype(str).str.zfill(2)
    iip_cols = [c for c in mospi_monthly.columns if "indices" in c.lower() or "growth" in c.lower() or "2019" in c or "2020" in c or "2021" in c]
    if iip_cols:
        mospi_clean = mospi_monthly[["year_month"] + iip_cols[:5]].copy()
    else:
        mospi_clean = mospi_monthly[["year_month"]].copy()
else:
    # Use annual growth rates
    mospi_clean = pd.DataFrame()
    print("  MOSPI: No monthly breakdown found — using annual data")

print(f"MOSPI IIP filtered: {mospi_clean.shape if len(mospi_clean) > 0 else 'empty'}")

# 3. MeitY: annual electronics production
meity = pd.read_csv(RAW / "rq1_meity_electronics.csv")
print(f"MeitY electronics: {meity.shape}")

# 4. Merge DGFT + HHI + MOSPI
rq1_panel = dgft_wide.merge(hhi_wide, on="year_month", how="left")
if len(mospi_clean) > 0 and "year_month" in mospi_clean.columns:
    rq1_panel = rq1_panel.merge(mospi_clean, on="year_month", how="left")

rq1_panel = rq1_panel.sort_values("year_month").reset_index(drop=True)

# Save
rq1_panel.to_csv(OUT / "rq1_panel.csv", index=False)
print(f"\n✓ RQ1 Panel saved: {OUT / 'rq1_panel.csv'}")
print(f"  Shape: {rq1_panel.shape}")
print(f"  Columns: {list(rq1_panel.columns)}")
print(f"  Date range: {rq1_panel['year_month'].min()} to {rq1_panel['year_month'].max()}")

# Comtrade as separate cross-country file
comtrade = pd.read_csv(RAW / "rq1_comtrade_crosscountry.csv")
comtrade.to_csv(OUT / "rq1_comtrade_benchmark.csv", index=False)
print(f"\n✓ RQ1 Comtrade benchmark: {comtrade.shape}")


---
## RQ2: Infrastructure — Spatial Join by State

**Goal:** 1 row per state (28 Indian states)  
**Columns:** state_name, ism_approved, energy_deficit_pct, td_losses_pct, installed_capacity, tariff, temperature, humidity

**Merge strategy:**
1. CEA (435 rows): extract state-level averages for deficit, T&D, capacity
2. SERC (240 rows): extract average industrial tariff per state
3. IMD Weather (8,768 rows): aggregate temp/humidity per facility location state
4. ISM Projects (6 rows): create ism_approved binary flag per state
5. Power Ministry (45 rows): national-level time series (supplementary)


In [ ]:
# ─── RQ2: Merge into state cross-section ───

# 1. CEA: extract key power metrics per state
cea = pd.read_csv(RAW / "rq2_cea_power_raw.csv", low_memory=False)
print(f"CEA raw: {cea.shape} ({cea.columns.tolist()[:5]}...)")

# Find state column
state_col = None
for c in ["state_u_ts_", "state", "region_state_u_t_", "state_system_region", "state_ut", "states_uts"]:
    if c in cea.columns:
        valid = cea[c].dropna()
        if len(valid) > 5:
            state_col = c
            break

if state_col:
    # Extract numeric columns that look like power metrics
    num_cols = cea.select_dtypes(include=[np.number]).columns.tolist()
    # Keep rows with valid state names (not null, not totals)
    cea_states = cea[cea[state_col].notna() & ~cea[state_col].str.contains("Total|All India|Region", case=False, na=False)].copy()
    
    # Group by state, take mean of numeric columns
    cea_agg = cea_states.groupby(state_col)[num_cols[:10]].mean().reset_index()
    cea_agg = cea_agg.rename(columns={state_col: "state_name"})
    print(f"CEA aggregated: {cea_agg.shape}")
else:
    cea_agg = pd.DataFrame()
    print("CEA: No state column found")

# 2. SERC tariffs: extract average tariff per state
serc = pd.read_csv(RAW / "rq2_serc_tariffs.csv", low_memory=False)
serc_state_col = None
for c in ["state_ut", "states_ut", "states__discoms"]:
    if c in serc.columns and serc[c].notna().sum() > 5:
        serc_state_col = c
        break

if serc_state_col and "average_cost_of_supply" in serc.columns:
    serc_agg = serc.groupby(serc_state_col)["average_cost_of_supply"].mean().reset_index()
    serc_agg = serc_agg.rename(columns={serc_state_col: "state_name", "average_cost_of_supply": "avg_cost_of_supply_rs_kwh"})
    print(f"SERC aggregated: {serc_agg.shape}")
else:
    serc_agg = pd.DataFrame()
    print("SERC: Using available columns")

# 3. ISM Projects: create ISM approval flag per state
ism = pd.read_csv(RAW / "ism_projects.csv")
ism_states = ism["state"].dropna().unique().tolist()
print(f"ISM approved states: {ism_states}")

# 4. Weather: average by nearest facility state
weather = pd.read_csv(RAW / "rq2_imd_weather.csv") if (RAW / "rq2_imd_weather.csv").exists() else pd.DataFrame()
locations = pd.read_csv(RAW / "rq2_semiconductor_facility_locations.csv") if (RAW / "rq2_semiconductor_facility_locations.csv").exists() else pd.DataFrame()

if len(weather) > 0 and "location" in weather.columns:
    weather_agg = weather.groupby("location").agg(
        avg_temp_c=("temperature_2m_mean", "mean"),
        avg_humidity_pct=("relative_humidity_2m_mean", "mean") if "relative_humidity_2m_mean" in weather.columns else ("temperature_2m_mean", "count"),
    ).reset_index()
    # Map location to state via locations reference
    if len(locations) > 0:
        weather_agg = weather_agg.merge(locations[["location", "state"]], on="location", how="left")
    print(f"Weather aggregated: {weather_agg.shape}")
else:
    weather_agg = pd.DataFrame()

# 5. Build final state panel
# Start with all 28 states
all_states = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhattisgarh",
    "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jharkhand", "Karnataka",
    "Kerala", "Madhya Pradesh", "Maharashtra", "Manipur", "Meghalaya", "Mizoram",
    "Nagaland", "Odisha", "Punjab", "Rajasthan", "Sikkim", "Tamil Nadu",
    "Telangana", "Tripura", "Uttar Pradesh", "Uttarakhand", "West Bengal"
]

rq2_panel = pd.DataFrame({"state_name": all_states})
rq2_panel["ism_approved"] = rq2_panel["state_name"].apply(
    lambda s: 1 if any(ism_s.lower() in s.lower() for ism_s in ism_states) else 0
)

# Merge CEA
if len(cea_agg) > 0:
    rq2_panel = rq2_panel.merge(cea_agg, on="state_name", how="left")

# Merge SERC
if len(serc_agg) > 0:
    rq2_panel = rq2_panel.merge(serc_agg, on="state_name", how="left")

rq2_panel.to_csv(OUT / "rq2_panel.csv", index=False)
print(f"\n✓ RQ2 Panel saved: {OUT / 'rq2_panel.csv'}")
print(f"  Shape: {rq2_panel.shape}")
print(f"  ISM approved: {rq2_panel['ism_approved'].sum()} states")
print(f"  Columns: {list(rq2_panel.columns)}")


---
## RQ3: Workforce — Cross-Country + India Panels

**Goal:** Two panels:
1. Cross-country: 1 row per country-year (5 countries × 10 years = ~50 rows)
2. India-specific: 1 row per state (for domestic workforce analysis)

**Merge strategy:**
1. UNESCO (49 rows) + World Bank talent (208 rows) + World Bank workforce (400 rows): join on country_iso3 + year
2. AICTE (116 rows) + PLFS (241 rows): join on state
3. NASSCOM (8 rows) + ISM/IESA (3 rows) + compiled workforce (13 rows): supplementary reference


In [ ]:
# ─── RQ3: Cross-country panel ───

# 1. UNESCO
unesco = pd.read_csv(RAW / "rq3_unesco_graduates.csv")
print(f"UNESCO: {unesco.shape}")

# 2. World Bank talent indicators
wb_talent = pd.read_csv(RAW / "rq3_worldbank_talent_indicators.csv") if (RAW / "rq3_worldbank_talent_indicators.csv").exists() else pd.DataFrame()
wb_work = pd.read_csv(RAW / "rq3_worldbank_workforce_indicators.csv") if (RAW / "rq3_worldbank_workforce_indicators.csv").exists() else pd.DataFrame()

# Pivot World Bank indicators to wide format
def pivot_wb(df, prefix):
    if len(df) == 0 or "indicator_id" not in df.columns:
        return pd.DataFrame()
    key_cols = [c for c in ["country", "iso3", "country_iso3"] if c in df.columns]
    iso_col = key_cols[0] if key_cols else None
    if iso_col is None:
        return pd.DataFrame()
    piv = df.pivot_table(index=[iso_col, "year"], columns="indicator_id", values="value", aggfunc="first").reset_index()
    piv.columns.name = None
    # Rename ISO column for consistency
    if iso_col != "country_iso3":
        piv = piv.rename(columns={iso_col: "country_iso3"})
    return piv

wb_t_wide = pivot_wb(wb_talent, "talent")
wb_w_wide = pivot_wb(wb_work, "work")
print(f"WB talent pivoted: {wb_t_wide.shape}")
print(f"WB workforce pivoted: {wb_w_wide.shape}")

# Merge cross-country panel
rq3_cc = unesco.copy()
if "country_iso3" in rq3_cc.columns:
    if len(wb_t_wide) > 0 and "country_iso3" in wb_t_wide.columns:
        rq3_cc = rq3_cc.merge(wb_t_wide, on=["country_iso3", "year"], how="outer")
    if len(wb_w_wide) > 0 and "country_iso3" in wb_w_wide.columns:
        rq3_cc = rq3_cc.merge(wb_w_wide, on=["country_iso3", "year"], how="outer", suffixes=("", "_wb"))

rq3_cc.to_csv(OUT / "rq3_crosscountry_panel.csv", index=False)
print(f"\n✓ RQ3 Cross-country panel: {rq3_cc.shape}")
print(f"  Countries: {sorted(rq3_cc['country_iso3'].dropna().unique()) if 'country_iso3' in rq3_cc.columns else 'N/A'}")

# ─── RQ3: India-specific panel ───
# AICTE intake
aicte = pd.read_csv(RAW / "rq3_aicte_intake.csv", low_memory=False) if (RAW / "rq3_aicte_intake.csv").exists() else pd.DataFrame()

# Supplementary reference datasets
nasscom = pd.read_csv(RAW / "rq3_nasscom_gcc_supply.csv") if (RAW / "rq3_nasscom_gcc_supply.csv").exists() else pd.DataFrame()
iesa = pd.read_csv(RAW / "rq3_ism_iesa_demand_projections.csv") if (RAW / "rq3_ism_iesa_demand_projections.csv").exists() else pd.DataFrame()
compiled = pd.read_csv(RAW / "rq3_semiconductor_workforce_compiled.csv") if (RAW / "rq3_semiconductor_workforce_compiled.csv").exists() else pd.DataFrame()

# Save India panel (AICTE is the primary)
if len(aicte) > 0:
    aicte.to_csv(OUT / "rq3_india_panel.csv", index=False)
    print(f"\n✓ RQ3 India panel (AICTE): {aicte.shape}")

# Save reference datasets
for name, df in [("nasscom", nasscom), ("iesa_projections", iesa), ("workforce_compiled", compiled)]:
    if len(df) > 0:
        df.to_csv(OUT / f"rq3_{name}_reference.csv", index=False)
        print(f"  + Reference: rq3_{name}_reference.csv ({len(df)} rows)")


---
## RQ4: Cost Competitiveness — Country-Year Panel Join

**Goal:** 1 row per country-year (9 countries × 10 years = ~90 rows)  
**Columns:** country_iso3, year, tax_rate, lending_rate, fdi, gdp_per_capita, electricity_price, subsidy_rate

**Merge strategy:**
1. World Bank cost (80 rows): primary panel backbone on country + year
2. Tax rates (10 rows): join on country_iso3 (single year cross-section)
3. Electricity prices (8 rows + 272 WB): join on country + year
4. RBI (19 rows): India-specific lending rates, join on year
5. DPIIT FDI (22 rows): India-specific FDI, join on year
6. Subsidies (6 + 4 + 6 rows): reference tables


In [ ]:
# ─── RQ4: Country-year panel ───

# 1. World Bank cost indicators (primary backbone)
wb_cost = pd.read_csv(RAW / "rq4_worldbank_cost.csv")
print(f"WB cost: {wb_cost.shape}")

# Standardize country column
iso_col = None
for c in ["economy", "Economy", "country_iso3"]:
    if c in wb_cost.columns:
        iso_col = c
        break
if iso_col and iso_col != "country_iso3":
    wb_cost = wb_cost.rename(columns={iso_col: "country_iso3"})

# 2. Tax rates
tax = pd.read_csv(RAW / "rq4_corporate_tax_rates_comparison.csv")
# Tax is single-year; add to latest year in panel
tax_slim = tax[["iso3", "statutory_rate_pct", "effective_rate_pct"]].rename(
    columns={"iso3": "country_iso3", "statutory_rate_pct": "corp_tax_statutory_pct",
             "effective_rate_pct": "corp_tax_effective_pct"})

# 3. Electricity prices
elec_ref = pd.read_csv(RAW / "rq4_electricity_prices_reference.csv") if (RAW / "rq4_electricity_prices_reference.csv").exists() else pd.DataFrame()
wb_elec = pd.read_csv(RAW / "rq4_worldbank_electricity_indicators.csv") if (RAW / "rq4_worldbank_electricity_indicators.csv").exists() else pd.DataFrame()

# Pivot WB electricity if available
if len(wb_elec) > 0 and "indicator_id" in wb_elec.columns:
    iso_c = [c for c in ["country", "iso3", "country_iso3"] if c in wb_elec.columns]
    if iso_c:
        wb_elec_wide = wb_elec.pivot_table(index=[iso_c[0], "year"], columns="indicator_id", values="value", aggfunc="first").reset_index()
        wb_elec_wide.columns.name = None
        if iso_c[0] != "country_iso3":
            wb_elec_wide = wb_elec_wide.rename(columns={iso_c[0]: "country_iso3"})
    else:
        wb_elec_wide = pd.DataFrame()
else:
    wb_elec_wide = pd.DataFrame()

# 4. RBI lending (India only)
rbi = pd.read_csv(RAW / "rq4_rbi_lending.csv") if (RAW / "rq4_rbi_lending.csv").exists() else pd.DataFrame()
if len(rbi) > 0:
    rbi["country_iso3"] = "IND"
    # Extract year as integer
    rbi["year_int"] = rbi["year"].astype(str).str[:4].astype(int)
    rbi_slim = rbi[["country_iso3", "year_int", "lending_rate", "call_notice_money_rate"]].rename(
        columns={"year_int": "year", "lending_rate": "rbi_lending_rate", "call_notice_money_rate": "rbi_call_rate"})

# 5. DPIIT FDI (India only)
fdi = pd.read_csv(RAW / "rq4_dpiit_fdi.csv") if (RAW / "rq4_dpiit_fdi.csv").exists() else pd.DataFrame()
if len(fdi) > 0:
    fdi["country_iso3"] = "IND"
    fdi["year_int"] = fdi["year"].astype(str).str[:4].astype(int)
    fdi_slim = fdi[["country_iso3", "year_int", "gross_fdi_inflows_usd_mn", "net_fdi_to_india_usd_mn"]].rename(
        columns={"year_int": "year"})

# Build panel
rq4_panel = wb_cost.copy()

# Merge tax rates (broadcast to all years for each country)
if len(tax_slim) > 0:
    rq4_panel = rq4_panel.merge(tax_slim, on="country_iso3", how="left")

# Merge WB electricity
if len(wb_elec_wide) > 0 and "country_iso3" in wb_elec_wide.columns and "year" in wb_elec_wide.columns:
    rq4_panel = rq4_panel.merge(wb_elec_wide, on=["country_iso3", "year"], how="left")

# Merge RBI (India rows only)
if len(rbi) > 0 and "year" in rq4_panel.columns:
    rq4_panel = rq4_panel.merge(rbi_slim, on=["country_iso3", "year"], how="left")

# Merge DPIIT FDI (India rows only)
if len(fdi) > 0 and "year" in rq4_panel.columns:
    rq4_panel = rq4_panel.merge(fdi_slim, on=["country_iso3", "year"], how="left")

rq4_panel.to_csv(OUT / "rq4_panel.csv", index=False)
print(f"\n✓ RQ4 Panel saved: {OUT / 'rq4_panel.csv'}")
print(f"  Shape: {rq4_panel.shape}")
print(f"  Countries: {sorted(rq4_panel['country_iso3'].dropna().unique()) if 'country_iso3' in rq4_panel.columns else 'N/A'}")
print(f"  Year range: {rq4_panel['year'].min()} to {rq4_panel['year'].max()}" if 'year' in rq4_panel.columns else "")
print(f"  Columns: {list(rq4_panel.columns)}")

# Save subsidy reference tables
for fname in ["rq4_ism_project_subsidies.csv", "rq4_ism_scheme_parameters.csv", "rq4_global_subsidy_comparison.csv"]:
    src = RAW / fname
    if src.exists():
        df = pd.read_csv(src)
        df.to_csv(OUT / fname, index=False)
        print(f"  + Reference: {fname} ({len(df)} rows)")


---
## Final Summary


In [ ]:
# Summary of all analysis-ready files
print("=" * 70)
print("MERGE COMPLETE — ANALYSIS-READY DATASETS")
print("=" * 70)

print(f"\nStored on Google Drive: {OUT}")
print(f"(Files persist even if Colab disconnects)\n")

for f in sorted(OUT.glob("*.csv")):
    df = pd.read_csv(f, low_memory=False)
    nulls = df.isnull().sum().sum()
    null_pct = round(nulls / max(df.size, 1) * 100, 1)
    n = f.stem.lower()
    rq = "RQ1" if "rq1" in n else ("RQ2" if "rq2" in n else ("RQ3" if "rq3" in n else ("RQ4" if "rq4" in n else "?")))
    print(f"  {rq}  {f.name:45s} {len(df):>6} rows × {len(df.columns):>3} cols  {null_pct:>5.1f}% null")

print(f"\n{'=' * 70}")
print("Google Drive location:")
print(f"  {OUT}")
print(f"\nNEXT STEPS:")
print("  1. Data Cleaning (method-specific preprocessing)")
print("  2. Sample Size Verification (post-cleaning row counts vs min N)")
print("  3. Run Statistical & ML Analysis")
print(f"{'=' * 70}")
